# Unwarping Sudoku using edge detection and Hough transform

- In this assignments, we're going to extract a 9x9 Sudoku grid using edge detection and Hough transform.
- We will
  1. find the four outer lines of the grid using Canny edge detection and Hough transform,
  2. compute the outer line intersection in order to determine positions of the top-left, top-right, bottom-right and bottom-left corners,
  3. normalize the grid into standard coordinates similarly to the [image_warping](image_warping.ipynb) assignment,
  4. optionally, recognize the digits.

<figure class="image">
  <img src="../figures/hough_rectification-expected_corners_outputs.png" alt="" style="width: 12.8in;"/>
  <figcaption>Figure 1: Expected outputs.</figcaption>
</figure>

In [ ]:
import sys

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import scipy
import seaborn as sns
import skimage

if '..' not in sys.path:
    sys.path.append('..')
from tests import test_hough_rectification

In [ ]:
plt.rcParams['figure.constrained_layout.use'] = True
np.set_printoptions(threshold=20, edgeitems=10, linewidth=140, precision=3, suppress=True)

# Load `sudoku-alt3.jpg`

In [ ]:
rgb = skimage.io.imread('../data/sudoku-alt3.jpg')
rgb = skimage.util.img_as_float(rgb)
rgb = skimage.transform.rescale(rgb, 640 / rgb.shape[1], channel_axis=2)
gray = skimage.color.rgb2gray(rgb)

In [ ]:
plt.imshow(gray, cmap='gray', vmin=0, vmax=1);

# Task 1: Detect edges using the Canny method

In [ ]:
########################################
# TODO: implement

edges = ...

########################################

In [ ]:
plt.imshow(edges, cmap='gray', vmin=0, vmax=1);

In [ ]:
test_hough_rectification.TestCanny.eval(img=edges)

# Task 2: Find lines using the Hough transform

- Use the Hough transform to detect grid lines of the sudoku.
- Make sure the lines along the border of the grid are found.
- We'll need those later to find the 4 corners of the grid.
- Store the lines as a NumPy array `lines` with shape `(#lines, 2)`, in which each row is a tuple `(dist, angle)`.

In [ ]:
def draw_hough_lines(ax, rgb, hough_lines, color='r'):
    ax.imshow(rgb);
    ax.set_xlim(0, rgb.shape[1])
    ax.set_ylim(rgb.shape[0], 0)
    for dist, angle in hough_lines:
        y1 = dist / np.sin(angle) if angle not in (-np.pi, 0., np.pi) else 1e9
        x2 = dist / np.cos(angle) if angle not in (-np.pi / 2., np.pi / 2.) else 1e9
        ax.axline((0, y1 if not np.isinf(y1) else 1e9), (x2 if not np.isinf(x2) else 1e9, 0), color=color)

In [ ]:
########################################
# TODO: implement

lines = ...

########################################

In [ ]:
plt.imshow(rgb);
draw_hough_lines(plt.gca(), rgb, lines);

In [ ]:
test_hough_rectification.TestHoughLines.eval(lines=lines)

# Task 3: filter the outer lines of the grid

- In the upcoming step, we'll find the 4 corners of the grid as intersection of the outer lines.
- Your task is to implement the function `filter_outer_lines` that
  - will take all the detected lines from the previous step,
  - and return the outer lines.
- Recommended approach:
  - Separate horizontal and vertical lines.
  - Horizontal lines will have angles close to $\pm\pi/2$, vertical ones close to $0$ (details depend on whether you use scikit-image or OpenCV).
  - The top most line will be the horizontal line with the smallest distance from the origin.
  - The bottom-most line will be the horizontal line with the largest distance from the origin.
  - In a similar manner, you can identify the left-most and right-most vertical lines as well.

In [ ]:
def filter_outer_lines(lines: np.ndarray) -> np.ndarray:
    """
    Filter Hough lines to keep only the outermost ones (top, right, left, bottom).

    Args:
        lines: (N, 2) array of (distance, angle) Hough line parameters
    Returns:
        (4, 2) array of (distance, angle) for top, right, left, bottom lines (in this order)
    """
    ########################################
    # TODO: implement

    raise NotImplementedError

    ########################################

In [ ]:
########################################
# TODO: implement

out_lines = ...

########################################

In [ ]:
plt.imshow(rgb);
draw_hough_lines(plt.gca(), rgb, outer_lines);

In [ ]:
test_hough_rectification.TestOuterHoughLines.eval(outer_lines=outer_lines)

# Task 4: Find the grid corners by intersecting the outer lines

- Implement the function `line_intersect_hnf` that computes intersection of two lines in Hessian normal form (HNF).
- Use the function to find the 4 corners of the grid.
- Store the corners into an Numpy array `corners` of shape (4, 2).

**Line intersection**
- You can calculate an intersection $\bold{p} = [p_1, p_2, p_3]^\top$ of two lines $\bold{l}_1 = [a_1, b_1, c_1]^\top$ and $\bold{l}_2 = [a_2, b_2, c_2]^\top$ in *standard form* as
  $$
  \bold{p} = \bold{l}_1 \times \bold{l}_2  = \begin{bmatrix}
    b_1 \cdot c_2 - b_2 \cdot c_1 \\
    a_2 \cdot c_1 - a_1 \cdot c_2 \\
    a_1 \cdot b_2 - a_2 \cdot b_1
  \end{bmatrix}
  $$
  where
  - $\times$ denotes the [cross product](https://en.wikipedia.org/wiki/Cross_product) of the two vectors,
  - $[p_1, p_2, p_3]$ are the unnormalized homogennous corrdinates of the intersection point. To get the final cartesian 2D intersection corrdinates $(x, y)$, divide by $z$:
    $$
    [x, y] = \left[\frac{p_1}{p_3}, \frac{p_2}{p_3}\right]
    $$
- You can use the function [`numpy.cross`](https://numpy.org/doc/stable/reference/generated/numpy.cross.html) from the numpy library for computing the cross product.
- Remember that from the Hough transform we got the lines in the HNF  
  $$
  x \cdot \cos \theta + y \cdot \sin \theta - r = 0
  $$
  so you first need to convert into the standard $(a,b,c)$ form
  $$
  a \cdot x + b \cdot y + c = 0
  $$

In [ ]:
def line_intersect_hnf(
    line1: np.ndarray,
    line2: np.ndarray,
) -> tuple[float, float]:
    """
    Compute the intersection point between two lines given in Hessian normal form.
    
    Args:
        line1: (dist1, angle1) Hough line parameters
        line2: (dist2, angle2) Hough line parameters
    Returns:
        (x, y) cartesian coordinates of the intersection point between line1 and line2
    """
    ########################################
    # TODO: implement
    
    raise NotImplementedError

    ########################################
    
    return x, y

In [ ]:
########################################
# TODO: implement

corners = ...

########################################

In [ ]:
plt.imshow(rgb)
plt.plot(corners[:, 0], corners[:, 1], 'or', markersize=7);  # corners is 4x2 matrix of (x,y) positions

In [ ]:
test_hough_rectification.TestLineIntersection.eval(line_intersect_hnf_fn=line_intersect_hnf)

# Task 5: Rectify the sudoku grid into standard square frame

- Extract and align the sudoku grid to standard size and coordinates.
- You can use the code from [geometric_transformations](../lectures/geometric_transformations.ipynb) lecture and/or [image_warping](image_warping.ipynb) assignment.
- Store the result as a NumPy array `roi` with shape $(M, M)$. Choose appropriate $M$.

In [ ]:
########################################
# TODO: implement

roi = ...

########################################

In [ ]:
plt.imshow(roi);

In [ ]:
test_hough_rectification.TestROI.eval(img=roi)

# (Bonus) Task 5: recognize the digits

- Split the registered grid into 9x9 square windows.
- Classify each window into one of possible classes: `['nothing', 1, 2, 3, 4, 5, 6, 7, 8, 9]`.
- Recommended approach:
  1. Load template digit images from `../data/123456789.png` and reshape them into an array of 9x40x40 pixels. 
  2. Center crop each cell from the rectified sudoku image to form 9x9x40x40 array of square windows. By center cropping we throw away grid lines popping into individual cells because of imprecise alignment.
  3. Threshold both templates and cropped cells to produce white digits on a black background.
  4. Compare each cell to each template using cross correlation [`skimage.feature.match_template`](https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.match_template) and take the best match value as a result. We need something like cross correlation because the digits are not perfectly aligned.
  5. The recognized digit for each cell the one with the largest cross correlation coefficient.
  6. Cells that don't have enough white pixels contain no digit and thus are classified as `'nothing'` (e.g. class index 0).
- Visualize the results and compute accuracy.
- The correct results for the image are in the file `../tests/data/sudoku-alt3.csv`.
- You the minimum required accuracy is 80 %, but you should be able to achieve 100 %.

<figure class="image">
  <img src="../figures/hough_rectification-expected_recognition_outputs.png" alt="" style="width: 12.8in;"/>
  <figcaption>Figure 2: Example outputs. Left: rectified ROI, middle: cropped and preprocessed cells, right: recognition output.</figcaption>
</figure>

In [ ]:
########################################
# TODO: implement

raise NotImplementedError

########################################